# chess-bote NNUE training on Colab

Trains the NNUE on Colab's GPU, using the same `nnue/train.py` and `nnue/export.py` as on your own machine.

1. **Runtime → Change runtime type → T4 GPU** (or any GPU).
2. Run the cells from top to bottom. Everything you want to keep is saved to Google Drive, because Colab wipes `/content` when the session ends.

**Training data:** either upload a data file you made locally (`gen_data.py` / `gen_selfplay.py`) to the Drive folder below (recommended, since data generation needs CPU cores and Colab only has ~2), or generate a small set here with step 3.

## 1. Settings

In [ ]:
REPO_URL = "https://github.com/juandanielsg/chess-bote.git"
BRANCH   = "main"                      # the branch that has the --device support in train.py

DRIVE_DIR = "/content/drive/MyDrive/chess-bote"   # data in, nets out
DATA_NAME = "distill.txt"              # a file inside DRIVE_DIR (fen|cp  or  fen|wdl|cp)
LABEL     = "cp"                       # "cp" for gen_data.py (Stockfish), "wdl" for gen_selfplay.py
RUN_NAME  = "net_colab"                # writes <RUN_NAME>.pt / .bin to DRIVE_DIR

EPOCHS = 60
BATCH  = 16384                         # bigger batches use the GPU better; try 4096-32768
LR     = 0.01

# The hidden-layer width must match the C++ engine (NN_HIDDEN in src/cpp/core.cpp).
# If you change it here, change core.cpp on your machine to the same value and rebuild there.
HIDDEN = 256

## 2. Setup: GPU check, Drive, clone and build the C++ core

**The repo is private, so Colab needs a read-only GitHub token to clone it** (one-time setup):
1. On GitHub: **Settings → Developer settings → Personal access tokens → Fine-grained tokens → Generate new token**. Under *Repository access* choose **Only select repositories → chess-bote**, and under *Permissions* set **Contents: Read-only**. Give it an expiry date.
2. In Colab: click the **key icon** in the left sidebar → **Add new secret**, name it `GITHUB_TOKEN`, paste the token, and turn on **Notebook access**.

Colab Secrets belong to your Google account, not to the notebook, so the token isn't saved in the notebook or shared with it.


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "No GPU: Runtime -> Change runtime type"

from google.colab import drive
drive.mount("/content/drive")
import os
os.makedirs(DRIVE_DIR, exist_ok=True)

In [ ]:
%cd /content
!rm -rf chess-bote
# The repo is private: clone with a GitHub token stored in Colab Secrets as GITHUB_TOKEN
# (see the instructions above). It is sent as an HTTP header, so it never shows up in
# the output or in .git/config.
import base64, subprocess
try:
    from google.colab import userdata
    _tok = userdata.get("GITHUB_TOKEN")
except Exception as e:
    _tok = None
    print(f"No GITHUB_TOKEN secret available ({type(e).__name__}); trying without it.")
_auth = []
if _tok:
    _b64 = base64.b64encode(f"x-access-token:{_tok}".encode()).decode()
    _auth = ["-c", f"http.extraHeader=Authorization: Basic {_b64}"]
_r = subprocess.run(["git", *_auth, "clone", "-q", "--branch", BRANCH, REPO_URL, "chess-bote"],
                    capture_output=True, text=True)
del _tok, _auth
if _r.returncode != 0:
    raise RuntimeError("git clone failed. Check the GITHUB_TOKEN secret (and its notebook "
                       "access toggle) and that the token can read the repo.\n" + _r.stderr)
print("cloned", REPO_URL, "@", BRANCH)
%cd /content/chess-bote

if HIDDEN != 256:
    !sed -i "s/^HIDDEN = 256/HIDDEN = {HIDDEN}/" nnue/train.py nnue/nnue_infer.py
    !sed -i "s/constexpr int NN_HIDDEN = 256;/constexpr int NN_HIDDEN = {HIDDEN};/" src/cpp/core.cpp

# Builds src/cpp/core.cpp into chess_bote._core (scikit-build-core + nanobind). Takes ~1-2 min.
!pip install -q . tqdm numpy
!python -c "from chess_bote import native; print('native core OK:', native.HAS_NATIVE)"
!grep -q -- "--device" nnue/train.py && echo "train.py has --device" || echo "WARNING: this branch's train.py has no --device; push the updated train.py first"

## 3. (Optional) Generate data on Colab

Skip this if you uploaded data to Drive. Colab has few CPU cores, so keep `COUNT` small, or let it run while you do something else.

In [ ]:
GENERATE = None          # None = skip, "stockfish" = gen_data.py (cp labels), "selfplay" = gen_selfplay.py (wdl labels)
COUNT    = 50_000

if GENERATE == "stockfish":
    !apt-get -qq install -y stockfish > /dev/null
    !python nnue/gen_data.py --count {COUNT} --depth 8 --quiet-filter --max-spread 80         --stockfish /usr/games/stockfish --out {DRIVE_DIR}/{DATA_NAME}
    LABEL = "cp"
elif GENERATE == "selfplay":
    import os
    !python nnue/gen_selfplay.py --count {COUNT} --nodes 5000 --jobs {os.cpu_count()}         --out {DRIVE_DIR}/{DATA_NAME}
    LABEL = "wdl"

## 4. Train on the GPU

In [ ]:
# Copy to local disk first (Drive reads are slow). The parsed-feature cache (.cache.npz)
# is saved back to Drive, so later sessions skip parsing the FENs.
import os, shutil
src = f"{DRIVE_DIR}/{DATA_NAME}"
assert os.path.exists(src), f"put your data file at {src}"
os.makedirs("nnue/data", exist_ok=True)
local = f"nnue/data/{DATA_NAME}"
shutil.copy(src, local)
if os.path.exists(src + ".cache.npz"):
    shutil.copy(src + ".cache.npz", local + ".cache.npz")
    os.utime(local + ".cache.npz")         # keep the cache newer than the data so it is reused
print(sum(1 for _ in open(local)), "positions")

In [ ]:
!python nnue/train.py --data {local} --label {LABEL} --device auto     --epochs {EPOCHS} --batch {BATCH} --lr {LR} --out {DRIVE_DIR}/{RUN_NAME}.pt 2>&1 | grep -E "^(device|loading|  [0-9]|epoch|best|saved)"

import shutil, os
if not os.path.exists(f"{DRIVE_DIR}/{DATA_NAME}.cache.npz"):
    shutil.copy(local + ".cache.npz", f"{DRIVE_DIR}/{DATA_NAME}.cache.npz")

## 5. Quantize and export to the engine format (`.bin`)

In [ ]:
!cd nnue && python export.py --in {DRIVE_DIR}/{RUN_NAME}.pt --out {DRIVE_DIR}/{RUN_NAME}.bin

## 6. Quick check: load the net into the C++ engine and search

This only checks that the net loads and that the engine plays with it. To find out whether it's *stronger*, run the cutechess A/B match from `nnue/BOOTSTRAP.md` on your own machine.

In [ ]:
from chess_bote import native
from chess_bote.board import Board
from chess_bote.native import _core
assert _core.load_nnue(f"{DRIVE_DIR}/{RUN_NAME}.bin"), "net failed to load (hidden size mismatch?)"
print("incremental accumulator selfcheck:", _core.nnue_selfcheck(Board().fen(), 3))
for fen in [Board().fen(),
            "r1bqkbnr/pppp1ppp/2n5/4p3/4P3/5N2/PPPP1PPP/RNBQKB1R w KQkq - 2 3",
            "4k3/8/8/8/8/8/8/4K2Q w - - 0 1"]:
    print(f"{_core.evaluate(fen):6d} cp (static, side to move) | search:", _core.search(fen, 8))

Download `RUN_NAME.bin` from your Drive folder, then on your machine:

```bash
printf 'uci
setoption name EvalFile value /path/to/net_colab.bin
isready
position startpos
go depth 10
quit
' | scripts/chessbote-uci.sh
```